# GSM8K answers cut by the token budget

**GPU: NVIDIA A100.** Run after `colab_run.ipynb`.

The scoring decodes at most 320 tokens per GSM8K answer. The base model and weakly attenuated adapters write long solutions, and part of them is cut before the final answer. This notebook decodes those answers again with a budget of 768 tokens (`scripts/extend_cut_outputs.py`), for every GSM8K score file at factors up to 1: the base model of each model, the GSM8K adapters (three seeds, the second model, the adapters saved at 10%, 20% and 40% of the steps) and the other adapters scored on GSM8K. Nothing is overwritten: the new rows go to `*__ext.csv` files next to the score files, and the analysis notebook uses them in place of the cut answers.

**Resumable**: after a disconnection, run all cells again. The last cell prints what changed.

## 0. Setup

In [ ]:
import os, subprocess, sys
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU or 'none')
assert GPU, 'No GPU: Runtime > Change runtime type'
EXPECTED = ('A100',)
ALLOW_OTHER_GPU = False   # set to True to run on another GPU anyway (it is recorded in run_info.json)
if not any(g in GPU for g in EXPECTED) and not ALLOW_OTHER_GPU:
    raise SystemExit(f'This notebook is planned for {EXPECTED} (bfloat16, 40 GB or more), got {GPU}: Runtime > Change runtime type')

REPO = '/content/dynalora'
BRANCH = 'paper'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/alexroehrs/dynalora.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log -1 --format='commit %H  %s'
!pip install -q -r requirements.txt
# Colab ships torchao 0.10, and peft 0.21 then refuses to load any LoRA adapter (it requires torchao > 0.16).
# Nothing in this repository uses torchao, so it is removed.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/dynalora_results'   # results root (Google Drive); results are not kept in the repository
L4 = f'{RESULTS}/level4'
os.makedirs(L4, exist_ok=True)
sys.path.insert(0, REPO)
from dynalora.runinfo import record_session
e = record_session(f'{L4}/run_info.json', 'GSM8K budget notebook start', seeds={'adapter_training': 'per adapter (s<seed> in its name)', 'train_subsets': 42, 'test_subsets': 0, 'bootstrap': 0})
print(e['gpu'], e.get('gpu_memory_gb'), e['git'], e['libraries'], sep='\n')
import torch
assert torch.cuda.is_bf16_supported(), 'this GPU has no bfloat16 support'
GRAD_CKPT = '--gradient_checkpointing' if torch.cuda.get_device_properties(0).total_memory < 60 * 2 ** 30 else ''   # 40 GB: checkpointing in training
print('gradient checkpointing in training:', bool(GRAD_CKPT))
FAILED = []   # steps that did not finish; reported at the end

In [ ]:
!python -m pytest -q tests/test_tasks.py -k "extend or greedy"

In [ ]:
def sh(cmd, log):
    print(cmd)
    get_ipython().system(f"bash -c 'set -o pipefail; {cmd} 2>&1 | tee -a {log}'")
    return get_ipython().user_ns.get('_exit_code', 1)

def step(name, cmd):
    rc = sh(cmd, f'{L4}/extend.log')
    if rc != 0:
        FAILED.append(name); print(f'WARNING: {name} did not finish (exit code {rc}); rerun the notebook to resume')
    return rc == 0

PRIMARY, SECOND = 'Qwen/Qwen2.5-7B-Instruct', 'microsoft/Phi-4-mini-instruct'
TASK, BUDGET = 'gsm8k', 768
OTHER = ['pubmedqa', 'yelp5', 'samsum']
A_DIR, S_DIR = f'{L4}/adapters', f'{L4}/scores'
tag = lambda m: m.rstrip('/').split('/')[-1]

def extend(model, adapter=None, adapter_task=None):
    """Base model (adapter None) or one adapter directory name."""
    cmd = (f'python scripts/extend_cut_outputs.py --model {model} --task {TASK} --base_cache {S_DIR}/_base '
           f'--max_new_tokens {BUDGET} --batch_size {32 if "mini" in model.lower() else 16}')
    if adapter:
        if not os.path.isdir(f'{S_DIR}/{adapter}'):
            print(f'{adapter}: no scores, skipped'); return True
        cmd += f' --adapter_dir {A_DIR}/{adapter} --out_dir {S_DIR}/{adapter}' + (f' --adapter_task {adapter_task}' if adapter_task else '')
    return step(adapter or f'{tag(model)} base', cmd)

## 1. Qwen2.5-7B-Instruct

In [ ]:
m = tag(PRIMARY)
if extend(PRIMARY):
    for name in [f'{m}__gsm8k__s42', f'{m}__gsm8k__s43', f'{m}__gsm8k__s44']:
        extend(PRIMARY, name)
    for t in OTHER:
        extend(PRIMARY, f'{m}__{t}__s42', adapter_task=t)
    for k in (10, 20, 40):
        extend(PRIMARY, f'{m}__gsm8k__s42__ckpt{k}')
print('not finished:', FAILED or 'nothing')

## 2. Phi-4-mini-instruct

In [ ]:
m = tag(SECOND)
if extend(SECOND):
    extend(SECOND, f'{m}__gsm8k__s42')
    for t in OTHER:
        extend(SECOND, f'{m}__{t}__s42', adapter_task=t)
print('not finished:', FAILED or 'nothing')

## 3. What changed

In [ ]:
import glob, pandas as pd
rows = []
for f in sorted(glob.glob(f'{S_DIR}/*/*__ext.csv')):
    e = pd.read_csv(f)
    for lam, g in e.groupby('factor'):
        rows.append(dict(file=f.replace(S_DIR + '/', ''), factor=lam, regenerated=len(g), now_end=int(g.ended.sum()),
                         score_of_these=round(100 * g.score.mean(), 1)))
print(pd.DataFrame(rows).to_string(index=False) if rows else 'nothing regenerated')
print('not finished:', FAILED or 'nothing')